# RoadVis — fine-tune the detector on Colab (GPU)

**RU.** Этот блокнот скачивает датасет с вашего сервера RoadVis Trainer, дообучает детектор (YOLO11s, imgsz 960,
rect, до 50 эпох с ранней остановкой) и отправляет `best.pt` обратно на сервер. Сервер сам сравнит новую модель с
текущей на замороженном тестовом наборе и опубликует её, только если она лучше.

1. *Среда выполнения → Сменить среду выполнения → GPU (T4)*.
2. Заполните поля ниже: адрес сервера и admin-токен (`ROADVIS_ADMIN_TOKEN`). Токен можно не вписывать, а сохранить в
   *Secrets* (значок ключа слева) под именем `ROADVIS_ADMIN_TOKEN`.
3. *Среда выполнения → Выполнить все*. Обучение занимает ~0.5–3 ч в зависимости от размера датасета.

**EN.** This notebook downloads the dataset from your RoadVis Trainer server, fine-tunes the detector (YOLO11s,
imgsz 960, rect, up to 50 epochs with early stopping) and uploads `best.pt` back. The server evaluates it against the
current model on the frozen test set and publishes it only if it is better.

1. *Runtime → Change runtime type → GPU (T4)*.
2. Fill in the server URL and the admin token (`ROADVIS_ADMIN_TOKEN`) below, or store the token in Colab *Secrets*
   (key icon on the left) as `ROADVIS_ADMIN_TOKEN`.
3. *Runtime → Run all*. Training takes about 0.5–3 h depending on the dataset size.

Never share a notebook with the token filled in. / Не делитесь блокнотом с вписанным токеном.

In [ ]:
#@title Settings / Настройки
SERVER_URL = "https://trafficcounter.uz/roadvis"  #@param {type:"string"}
ADMIN_TOKEN = ""  #@param {type:"string"}
START_FROM = "auto"  #@param ["auto", "yolo11s.pt"]
EPOCHS = 50  #@param {type:"integer"}
PATIENCE = 10  #@param {type:"integer"}
IMGSZ = 960  #@param {type:"integer"}
UPLOAD_RESULT = True  #@param {type:"boolean"}

In [ ]:
#@title Install / Установка
!nvidia-smi -L || echo "No GPU: Runtime → Change runtime type → GPU"
%pip install -q "ultralytics>=8.4,<9" requests pyyaml

In [ ]:
#@title Download the dataset / Скачать датасет
import json, os, pathlib, shutil, zipfile
import requests, yaml

if not ADMIN_TOKEN:
    try:
        from google.colab import userdata
        ADMIN_TOKEN = userdata.get("ROADVIS_ADMIN_TOKEN") or ""
    except Exception:
        pass
assert ADMIN_TOKEN, "Set ADMIN_TOKEN (or the Colab secret ROADVIS_ADMIN_TOKEN)"
BASE = SERVER_URL.rstrip("/")
H = {"Authorization": f"Bearer {ADMIN_TOKEN}"}
WORK = pathlib.Path("/content/roadvis")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

# The frozen test set stays on the server (it is used only by the quality gate).
with requests.get(f"{BASE}/api/v1/admin/dataset/export", params={"include_test": "false"}, headers=H,
                  stream=True, timeout=1800) as r:
    r.raise_for_status()
    with open(WORK / "dataset.zip", "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
with zipfile.ZipFile(WORK / "dataset.zip") as z:
    z.extractall(WORK / "dataset")
(WORK / "dataset.zip").unlink()

cfg = yaml.safe_load(open(WORK / "dataset" / "data.yaml"))
cfg["path"] = str(WORK / "dataset")
cfg.pop("test", None)
DATA = WORK / "data.yaml"
yaml.safe_dump(cfg, open(DATA, "w"), sort_keys=False)
N_TRAIN = len(list((WORK / "dataset" / "images" / "train").glob("*.jpg")))
N_VAL = len(list((WORK / "dataset" / "images" / "val").glob("*.jpg")))
print(f"train images: {N_TRAIN}, val images: {N_VAL}, classes: {cfg['names']}")
assert N_TRAIN > 0, "The dataset is empty — let the phones collect frames first."

In [ ]:
#@title Starting weights / Начальные веса
INIT, INIT_NAME = "yolo11s.pt", "yolo11s.pt"
if START_FROM == "auto":
    r = requests.get(f"{BASE}/api/v1/admin/model/current", headers=H, timeout=600)
    if r.status_code == 200:
        (WORK / "current.pt").write_bytes(r.content)
        INIT = str(WORK / "current.pt")
        INIT_NAME = "roadvis-" + r.headers.get("X-RoadVis-Version", "current")
    elif r.status_code != 404:
        r.raise_for_status()
print("fine-tuning from", INIT_NAME)

In [ ]:
#@title Train / Обучение
import torch
from ultralytics import YOLO

model = YOLO(INIT)
results = model.train(data=str(DATA), imgsz=IMGSZ, rect=True, epochs=EPOCHS, patience=PATIENCE, batch=-1,
                      device=0 if torch.cuda.is_available() else "cpu", project=str(WORK / "runs"), name="roadvis",
                      exist_ok=True, plots=False)
RUN_DIR = pathlib.Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
print("best weights:", BEST, f"{BEST.stat().st_size / 1e6:.1f} MB")

In [ ]:
#@title Upload the candidate / Отправить кандидата на сервер
import datetime, time
import ultralytics

val = {}
try:
    val = {k: float(v) for k, v in (results.results_dict or {}).items()}
except Exception:
    pass
metrics = {
    "baseModel": "yolo11s",
    "initWeights": INIT_NAME,
    "imgsz": IMGSZ, "epochs": EPOCHS, "patience": PATIENCE,
    "epochsRun": int(getattr(model.trainer, "epoch", -1)) + 1,
    "bestFitness": float(getattr(model.trainer, "best_fitness", 0) or 0),
    "trainImages": N_TRAIN, "valImages": N_VAL,
    "val": val,
    "ultralytics": ultralytics.__version__,
    "trainedAt": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
print(json.dumps(metrics, indent=2))
if UPLOAD_RESULT:
    with open(BEST, "rb") as f:
        r = requests.post(f"{BASE}/api/v1/admin/candidates", headers=H, timeout=3600,
                          files={"weights": ("best.pt", f, "application/octet-stream"),
                                 "metrics": ("metrics.json", json.dumps(metrics), "application/json")})
    print(r.status_code, r.text)
    r.raise_for_status()
    CANDIDATE = r.json()["id"]
    print("Candidate", CANDIDATE, "queued. The server worker now runs the quality gate on CPU.")

In [ ]:
#@title (optional) Wait for the gate result / Дождаться решения сервера
for _ in range(240):
    s = requests.get(f"{BASE}/api/v1/admin/candidates/{CANDIDATE}", headers=H, timeout=60).json()
    if s["status"] not in ("pending", "evaluating"):
        break
    time.sleep(60)
print(s["status"], s.get("message"))
decision = (s.get("report") or {}).get("decision")
if decision:
    for c in decision.get("checks", []):
        print(c)